Try and Test Here

In [1]:
import os
import sys
from pathlib import Path

# Notebook is inside .../PruningNAS/PruningNAS, so add project root (.../PruningNAS)
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
	sys.path.insert(0, str(project_root))

# Install local package in editable mode (run once; safe to re-run)
%pip install -e ..

import torch
from PruningNAS.Models.DenseNet import *
from PruningNAS.Models.ResNetBasic import *
from PruningNAS.Models.ResNetBottleNeck import *
from PruningNAS.Utills.PrunUtillCP import ChannelPrunner, channel_prune_resnet

Obtaining file:///D:/Sutanu_WorkSpace/PruningNas
Note: you may need to restart the kernel to use updated packages.


ERROR: file:///D:/Sutanu_WorkSpace/PruningNas does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


Device: cuda


In [29]:
# Initialize DenseNet-121 model
model =DenseNet121(classes=10)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#       Load pre-trained weights (if available)
# model_path = r'PruningNAS\checkpoint\Resnet-34\Resnet-34_cifar_95.69000244140625.pth'
# model = torch.load(model_path, map_location=torch.device(device),weights_only=False)  # Use 'cpu' if necessary

model.to(device)
model.eval()

# P_model=channel_prune_resnet(model, 0.7)
# P_model.eval()
# P_model.to(device)
count=0

prunable_weights = []
print("Pruned Model Architecture:")
prunable_weights.append(model.init_conv)
for blocks in model.blocks:
    if isinstance(blocks, DenseBlock):
       for name, module in blocks.named_children():
           for sub_name, sub_module in module.named_children():
               prunable_weights.append(sub_module.conv1)
               prunable_weights.append(sub_module.conv2)


print("Total prunable layers in DenseNet-121:", len(prunable_weights))


# for blocks in model.blocks:
#     DenseBlock
#     TransitionLayer


print("Total prunable layers in DenseNet-121:", count)


model_path=r'PruningNAS\checkpoint\Resnet-34\FGP'
model_dir = os.path.dirname(model_path)



print("dir", model_dir)
# # Example: create dummy input data (batch size 1, 3 channels, 32x32 for CIFAR-10)
# dummy_input = torch.randn(1, 3, 32, 32)

# # Forward pass through the pruned model
# with torch.no_grad():
#     dummy_output = P_model(dummy_input.to(device))      
# print("\nOutput shape:", dummy_output.shape)
# print("Output:", dummy_output)

# # Calculate pruning ratio count for DenseNet: conv0 + num_denseblocks + num_transitions
# num_denseblocks = 4  # DenseNet121 has 4 dense blocks
# num_transitions = 3  # transitions between blocks (not after last block)
# prune_ratio_count = 1 + num_denseblocks + num_transitions

# # Create uniform pruning ratios (0.7 = 30% pruning)
# sparsity_dict = [0.7] * prune_ratio_count

# # Prune the model using ChannelPrunner
# pruned_model = ChannelPrunner(model, sparsity_dict, 'Densenet')
# pruned_model.eval()

# # Test the pruned model with dummy input
# print("Pruned Model Architecture:")
# for name, module in pruned_model.named_modules():
#     if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear, torch.nn.BatchNorm2d)):
#         params = list(module.parameters())
#         if params:
#             print(f"{name}: {params[0].shape}")

# # Forward pass
# dummy_output = pruned_model(dummy_input)
# print("\nOutput shape:", dummy_output.shape)
# print("Output:", dummy_output)

Pruned Model Architecture:
Total prunable layers in DenseNet-121: 117
Total prunable layers in DenseNet-121: 0
dir PruningNAS\checkpoint\Resnet-34


In [39]:
count=0
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Conv2d):
        parent_name = name.rsplit(".", 1)[0] if "." in name else "<root>"
        if parent_name[:-2] == 'blocks':
            continue
        print(f"{name}: {module}")
        count += 1
print(f"Total prunable layers: {count}")

init_conv.0: Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
blocks.0.block.0.conv1: Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
blocks.0.block.0.conv2: Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
blocks.0.block.1.conv1: Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
blocks.0.block.1.conv2: Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
blocks.0.block.2.conv1: Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
blocks.0.block.2.conv2: Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
blocks.0.block.3.conv1: Conv2d(160, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
blocks.0.block.3.conv2: Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
blocks.0.block.4.conv1: Conv2d(192, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
blocks.0.block.4.conv2: Conv2d(128, 32, kernel_size=(3, 3), 

In [30]:
count=1

for blocks in model.blocks:
    if isinstance(blocks, DenseBlock):
       for name, module in blocks.named_children():
           for sub_name, sub_module in module.named_children():
               count+=1
               print(f"DenseBlock {count}: {sub_name} - {sub_module.conv1.weight.shape}, {sub_module.conv2.weight.shape}, ")
               

    elif isinstance(blocks, TransitionLayer):
       print(f"TransitionLayer {count}:")

print('----------------------------')

print("Total prunable layers in DenseNet-121:", count)
print("Total prunable layers in DenseNet-121:", 6+12+32+48+1)

DenseBlock 2: 0 - torch.Size([128, 64, 1, 1]), torch.Size([32, 128, 3, 3]), 
DenseBlock 3: 1 - torch.Size([128, 96, 1, 1]), torch.Size([32, 128, 3, 3]), 
DenseBlock 4: 2 - torch.Size([128, 128, 1, 1]), torch.Size([32, 128, 3, 3]), 
DenseBlock 5: 3 - torch.Size([128, 160, 1, 1]), torch.Size([32, 128, 3, 3]), 
DenseBlock 6: 4 - torch.Size([128, 192, 1, 1]), torch.Size([32, 128, 3, 3]), 
DenseBlock 7: 5 - torch.Size([128, 224, 1, 1]), torch.Size([32, 128, 3, 3]), 
TransitionLayer 7:
DenseBlock 8: 0 - torch.Size([128, 128, 1, 1]), torch.Size([32, 128, 3, 3]), 
DenseBlock 9: 1 - torch.Size([128, 160, 1, 1]), torch.Size([32, 128, 3, 3]), 
DenseBlock 10: 2 - torch.Size([128, 192, 1, 1]), torch.Size([32, 128, 3, 3]), 
DenseBlock 11: 3 - torch.Size([128, 224, 1, 1]), torch.Size([32, 128, 3, 3]), 
DenseBlock 12: 4 - torch.Size([128, 256, 1, 1]), torch.Size([32, 128, 3, 3]), 
DenseBlock 13: 5 - torch.Size([128, 288, 1, 1]), torch.Size([32, 128, 3, 3]), 
DenseBlock 14: 6 - torch.Size([128, 320, 1,

In [1]:
count=1

print(f"Pruned Model Architecture: {model.block_config}")
for i, num_layers in enumerate(model.block_config):
    print(model.blocks[i])
    count+=1
    if i != len(model.block_config) - 1:
        print(f"TransitionLayer {count}:")

               

print('----------------------------')

print("Total prunable layers in DenseNet-121:", count)
print("Total prunable layers in DenseNet-121:", 6+12+32+48+1)

NameError: name 'model' is not defined

In [ ]:
import torch
from sympy import Union

from PruningNAS.Models.DenseNet import DenseBlock, TransitionLayer
from PruningNAS.Utills.PrunUtillCP import count_densenet_blocks, get_num_channels_to_keep


@torch.no_grad()
def channel_prune_densenet(model, prune_ratios: Union[float, dict, list]):
    """Prune DenseNet model by channel pruning with specified ratios."""
    
    def prune_dense_layer(layer, prune_ratio1, prune_ratio2, prev_n_keep):
        """Prune a single DenseLayer in the DenseBlock."""
        # Prune norm1 and conv1 based on input channels
        layer.norm1.weight.set_(layer.norm1.weight.detach()[:prev_n_keep])
        layer.norm1.bias.set_(layer.norm1.bias.detach()[:prev_n_keep])
        layer.norm1.running_mean.set_(layer.norm1.running_mean.detach()[:prev_n_keep])
        layer.norm1.running_var.set_(layer.norm1.running_var.detach()[:prev_n_keep])
        layer.conv1.weight.set_(layer.conv1.weight.detach()[:, :prev_n_keep])

        # Calculate kept channels after conv1
        original_channels = layer.conv1.out_channels
        n_keep_conv1 = get_num_channels_to_keep(original_channels, prune_ratio1)
        layer.conv1.weight.set_(layer.conv1.weight.detach()[:n_keep_conv1])

        # Prune norm2 and conv2 based on conv1 output
        layer.norm2.weight.set_(layer.norm2.weight.detach()[:n_keep_conv1])
        layer.norm2.bias.set_(layer.norm2.bias.detach()[:n_keep_conv1])
        layer.norm2.running_mean.set_(layer.norm2.running_mean.detach()[:n_keep_conv1])
        layer.norm2.running_var.set_(layer.norm2.running_var.detach()[:n_keep_conv1])
        layer.conv2.weight.set_(layer.conv2.weight.detach()[:, :n_keep_conv1])

        # Calculate kept channels after conv2 (output channels)
        original_channels = layer.conv2.out_channels
        n_keep_conv2 = get_num_channels_to_keep(original_channels, prune_ratio2)
        layer.conv2.weight.set_(layer.conv2.weight.detach()[:n_keep_conv2])
        
        return n_keep_conv2

    def prune_transition_layer(transition, prev_n_keep, prune_ratio):
        """Prune a transition layer."""
        transition.norm.weight.set_(transition.norm.weight.detach()[:prev_n_keep])
        transition.norm.bias.set_(transition.norm.bias.detach()[:prev_n_keep])
        transition.norm.running_mean.set_(transition.norm.running_mean.detach()[:prev_n_keep])
        transition.norm.running_var.set_(transition.norm.running_var.detach()[:prev_n_keep])
        transition.conv.weight.set_(transition.conv.weight.detach()[:, :prev_n_keep])

        # Calculate kept channels after transition
        original_channels = transition.conv.out_channels
        n_keep = get_num_channels_to_keep(original_channels, prune_ratio)
        transition.conv.weight.set_(transition.conv.weight.detach()[:n_keep])

        return n_keep

    # Validate and normalize prune_ratios
    assert isinstance(prune_ratios, (float, dict, list))
    
    if isinstance(prune_ratios, dict):
        prune_ratios = list(prune_ratios.values())
    elif isinstance(prune_ratios, float):
        prune_ratios = [prune_ratios] * count_densenet_blocks(model)
    
    assert len(prune_ratios) == count_densenet_blocks(model), \
        f"Expected {count_densenet_blocks(model)} prune ratios, got {len(prune_ratios)}"

    # Prune initial convolution
    original_channels = model.init_conv[0].out_channels
    n_keep = get_num_channels_to_keep(original_channels, prune_ratios[0])
    model.init_conv[0].weight.set_(model.init_conv[0].weight.detach()[:n_keep])
    model.init_conv[1].weight.set_(model.init_conv[1].weight.detach()[:n_keep])
    model.init_conv[1].bias.set_(model.init_conv[1].bias.detach()[:n_keep])
    model.init_conv[1].running_mean.set_(model.init_conv[1].running_mean.detach()[:n_keep])
    model.init_conv[1].running_var.set_(model.init_conv[1].running_var.detach()[:n_keep])

    # Prune DenseBlocks and TransitionLayers
    ratio_idx = 1  # Start from index 1 (init_conv uses index 0)
    
    for block in model.blocks:
        if isinstance(block, DenseBlock):
            for layer in block.block:
                if ratio_idx + 1 < len(prune_ratios):
                    n_keep = prune_dense_layer(layer, prune_ratios[ratio_idx], 
                                               prune_ratios[ratio_idx + 1], n_keep)
                    ratio_idx += 1
        elif isinstance(block, TransitionLayer):
            if ratio_idx < len(prune_ratios):
                n_keep = prune_transition_layer(block, n_keep, prune_ratios[ratio_idx])
                ratio_idx += 1

    # Prune final fully connected layer
    model.fc.weight.set_(model.fc.weight.detach()[:, :n_keep])

    return model

## FGP Testing area


In [2]:
import copy
from multiprocessing.dummy import freeze_support
import pickle

import torch

from PruningNAS.DataProcess.DataPreprocessing import get_dataloaders
from PruningNAS.Models.VGG import VGG
from PruningNAS.Models.DenseNet import DenseNet121
from PruningNAS.Models.MobileNetV1 import MobileNetV1
from PruningNAS.Models.ResNetBasic import *
from PruningNAS.Models.ResNetBottleNeck import *
from PruningNAS.PruneEvaluator import load_and_print_accuracies
from PruningNAS.Utills.PrunUtillCP import apply_channel_sorting, channel_prune_densenet, channel_prune_resnet
from PruningNAS.Utills.PrunUtillFGP import FineGrainedPruner, get_sparsity_dict_template_FGP
from PruningNAS.Utills.TrainingModulesUtills import evaluate


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# Initialize the model
path='../dataset/cifar10'
model=MobileNetV1(classes=10)
model_path=r'.\checkpoint\MobilenetV1\MobilenetV1_cifar_94.129997.pth'
model = torch.load(model_path, map_location=torch.device(device),weights_only=False)  # Use 'cpu' if necessary

model.to(device)

sorted_model=copy.deepcopy(model)
sparsity_dict = get_sparsity_dict_template_FGP(sorted_model, value=0.1)
pruner = FineGrainedPruner(sorted_model, sparsity_dict)

sorted_model.eval()  # Set to evaluation mode
model.eval()  # Set to evaluation modeS

dummy_input = torch.randn(1, 3, 32, 32).to(device)

print("Original Model Architecture:")
for name, module in model.named_modules():
    if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear, torch.nn.BatchNorm2d)):
        params = list(module.parameters())
        if params:
            print(f"{name}: {params[0].shape}")
            zero_count = (params[0] == 0).sum().item()
            total_count = params[0].numel()
            print(f"{name}: zeros={zero_count}/{total_count} ({100*zero_count/total_count:.2f}%)")

print("Sorted Model Architecture:")
for name, module in sorted_model.named_modules():
    if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear, torch.nn.BatchNorm2d)):
        params = list(module.parameters())
        if params:
            print(f"{name}: {params[0].shape}")
            zero_count = (params[0] == 0).sum().item()
            total_count = params[0].numel()
            print(f"{name}: zeros={zero_count}/{total_count} ({100*zero_count/total_count:.2f}%)")


model.to(device)
sorted_model.to(device)
with torch.no_grad():
    dummy_output = model(dummy_input)    
print("\nOutput shape:", dummy_output.shape)
print("Output:", dummy_output)


with torch.no_grad():
    dummy_output = sorted_model(dummy_input)    
print("\nOutput shape:", dummy_output.shape)
print("Output:", dummy_output)


# train_dataloader,test_dataloader=get_dataloaders(path )

# model_accuracy,_ =evaluate(model,test_dataloader)
# print("Original Model Accuracy:",model_accuracy)

# sorted_model_accuracy,_ =evaluate(sorted_model,test_dataloader)  
# print("Sorted Model Accuracy:",sorted_model_accuracy)





d:\Sutanu_WorkSpace\PruningNAS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda
Original Model Architecture:
model.0: torch.Size([32, 3, 3, 3])
model.0: zeros=0/864 (0.00%)
model.1: torch.Size([32])
model.1: zeros=0/32 (0.00%)
model.3.depthwise: torch.Size([32, 1, 3, 3])
model.3.depthwise: zeros=0/288 (0.00%)
model.3.bn1: torch.Size([32])
model.3.bn1: zeros=0/32 (0.00%)
model.3.pointwise: torch.Size([64, 32, 1, 1])
model.3.pointwise: zeros=0/2048 (0.00%)
model.3.bn2: torch.Size([64])
model.3.bn2: zeros=0/64 (0.00%)
model.4.depthwise: torch.Size([64, 1, 3, 3])
model.4.depthwise: zeros=0/576 (0.00%)
model.4.bn1: torch.Size([64])
model.4.bn1: zeros=0/64 (0.00%)
model.4.pointwise: torch.Size([128, 64, 1, 1])
model.4.pointwise: zeros=0/8192 (0.00%)
model.4.bn2: torch.Size([128])
model.4.bn2: zeros=0/128 (0.00%)
model.5.depthwise: torch.Size([128, 1, 3, 3])
model.5.depthwise: zeros=0/1152 (0.00%)
model.5.bn1: torch.Size([128])
model.5.bn1: zeros=0/128 (0.00%)
model.5.pointwise: torch.Size([128, 128, 1, 1])
model.5.pointwise: zeros=0/16384 (0.00%)
model.5.bn2: torch

## CP Testing Area

In [2]:
import copy
from multiprocessing.dummy import freeze_support
import pickle

import torch

from PruningNAS.DataProcess.DataPreprocessing import get_dataloaders
from PruningNAS.Models.DenseNet import DenseNet121
from PruningNAS.Models.MobileNetV1 import MobileNetV1
from PruningNAS.Models.ResNetBasic import *
from PruningNAS.Models.ResNetBottleNeck import *
from PruningNAS.PruneEvaluator import load_and_print_accuracies
from PruningNAS.Utills.PrunUtillCP import apply_channel_sorting, channel_prune_densenet, channel_prune_mobilenetv1, channel_prune_resnet
from PruningNAS.Utills.TrainingModulesUtills import evaluate


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(device)
    # Initialize the model
    path='../dataset/cifar10'
    model=MobileNetV1(classes=10)
    model_path=r'.\checkpoint\MobilenetV1\MobilenetV1_cifar_94.129997.pth'
    model = torch.load(model_path, map_location=torch.device(device),weights_only=False)  # Use 'cpu' if necessary

    model.to(device)

    train_dataloader,test_dataloader=get_dataloaders(path )
    dense_model_accuracy,_=evaluate(model,test_dataloader)
    print(f"Original model accuracy: {dense_model_accuracy:.4f}")

    sorted_model=copy.deepcopy(model)
    sorted_model.requires_grad_(False)  # Disable gradients before pruning
    sorted_model = apply_channel_sorting(sorted_model)
    # pr=[0.00,0.0,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0]
    sorted_model = channel_prune_mobilenetv1(sorted_model, 0.7)

    sorted_model.eval()  # Set to evaluation mode
    model.eval()  # Set to evaluation modeS

    dummy_input = torch.randn(20, 3, 32, 32).to(device)

    print("Original Model Architecture:")
    for name, module in model.named_modules():
        if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear, torch.nn.BatchNorm2d)):
            params = list(module.parameters())
            if params:
                print(f"{name}: {params[0].shape}")

    print("Sorted Model Architecture:")
    for name, module in sorted_model.named_modules():
        if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear, torch.nn.BatchNorm2d)):
            params = list(module.parameters())
            if params:
                print(f"{name}: {params[0].shape}")

    model.to(device)
    sorted_model.to(device)
    with torch.no_grad():
        dummy_output = model(dummy_input)    
    print("\nOutput shape:", dummy_output.shape)
    print("Output:", dummy_output)


    with torch.no_grad():
        dummy_output = sorted_model(dummy_input)    
    print("\nOutput shape:", dummy_output.shape)
    print("Output:", dummy_output)

    model_accuracy,_ =evaluate(model,test_dataloader)
    print("Original Model Accuracy:",model_accuracy)

    sorted_model_accuracy,_ =evaluate(sorted_model,test_dataloader)  
    print("Sorted Model Accuracy:",sorted_model_accuracy)



if __name__ == '__main__':
    freeze_support()
    main() 


d:\Sutanu_WorkSpace\PruningNAS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


d:\Sutanu_WorkSpace\PruningNAS\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Original model accuracy: 94.1300
DepthwiseSeparableConv(
  (depthwise): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=10, bias=False)
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pointwise): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)
DepthwiseSeparableConv(
  (depthwise): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=19, bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pointwise): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)
DepthwiseSeparableConv(
  (depthwise): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=38, bias=False)
  (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, t

Original Model Accuracy: 94.12999725341797


Sorted Model Accuracy: 9.999999046325684
